# 00 · Set up the GPU node

Run this first on a fresh machine. It installs dependencies, checks the GPU, and
pulls every data file the other notebooks need straight from GitHub — so nothing
has to be uploaded by hand.

**What you need on the node before running this:** `nb_common.py` and the
`notebooks/` folder. The simplest way to get both:

```bash
git clone https://github.com/SamAbr/PSA-MT.git
cd PSA-MT
jupyter lab --ip 0.0.0.0 --no-browser
```

Then open `notebooks/00_setup.ipynb`.

If you would rather not clone, copy `nb_common.py` and the notebook you want
onto the node; the data still downloads itself.

## 1. Dependencies

In [ ]:
import subprocess, sys

PACKAGES = [
    "torch", "transformers>=4.41", "sentencepiece", "accelerate",
    "datasets", "sacrebleu", "ftfy", "pandas", "matplotlib",
]

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import torch, transformers, sacrebleu, ftfy  # noqa: F401
    print("core packages already present")
except ImportError:
    print("installing ...")
    pip_install(PACKAGES)
    print("done - RESTART THE KERNEL before continuing")

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
sys.path.insert(0, str(pathlib.Path.cwd()))
import nb_common as C

C.set_seed()
print(f"project root : {C.ROOT}")
print(f"data source  : {C.RAW_BASE}")

## 2. Hardware

Size your batches against **free** VRAM, not total — the node is shared with
other jobs.

In [ ]:
C.gpu_report()

## 3. Pull the data from GitHub

Each file is downloaded once and cached under `data/`. Re-running this cell is
free. If a download 404s, the file has not been pushed yet, or
`C.GITHUB_BRANCH` does not match the branch it lives on.

In [ ]:
REQUIRED = ["bible_en_guz_swh.csv", "psa_ke_train.csv", "psa_ke_test.csv"]
OPTIONAL = ["lughayangu_sentences.csv", "english_psas.csv",
            "psa_parallel_dataset.csv", "psa_ke_test_en_guz.csv"]

print("required:")
for name in REQUIRED:
    C.download(name)

print("\noptional (EDA and reference only):")
for name in OPTIONAL:
    try:
        C.download(name)
    except FileNotFoundError as exc:
        print(f"  skipped {name}: {str(exc).strip().splitlines()[1]}")

## 4. Verify what arrived

In [ ]:
import pandas as pd

for name in REQUIRED + OPTIONAL:
    path = C.ROOT / C.REPO_PATHS[name]
    if not path.exists():
        print(f"  --  {name:<28} absent")
        continue
    try:
        df = pd.read_csv(path)
        cols = ", ".join(list(df.columns)[:5])
        print(f"  ok  {name:<28} {len(df):>7,} rows | {cols}")
    except Exception as exc:
        print(f"  !!  {name:<28} unreadable: {exc}")

In [ ]:
# Spot-check that the Ekegusii really is Ekegusii, using the same trigram
# classifier the preprocessing used. Catches a truncated or wrong-branch
# download before you spend GPU hours on it.
import csv, math
from collections import Counter
csv.field_size_limit(10 ** 7)

def trigrams(t, n=3):
    t = " " + " ".join(t.lower().split()) + " "
    return [t[i:i + n] for i in range(len(t) - n + 1)]

rows = list(csv.DictReader(open(C.BIBLE_CSV, encoding="utf-8")))[:6000]
models = {lang: Counter(g for r in rows for g in trigrams(r[col]))
          for lang, col in [("en", "english"), ("guz", "ekegusii"), ("swh", "swahili")]}
totals = {l: sum(c.values()) for l, c in models.items()}
V = len({g for c in models.values() for g in c})

def predict(text):
    return max(models, key=lambda l: sum(
        math.log((models[l].get(g, 0) + 0.1) / (totals[l] + 0.1 * V))
        for g in trigrams(text)) / max(1, len(trigrams(text))))

ke = list(csv.DictReader(open(C.PSA_KE_TRAIN_CSV, encoding="utf-8")))[:300]
ok = sum(predict(r["ekegusii"]) == "guz" for r in ke)
print(f"PSA_KE Ekegusii column verified: {ok}/{len(ke)} classified Ekegusii")
assert ok / len(ke) > 0.9, "data looks wrong - re-download"
print("data integrity OK")

## 5. Next

| Notebook | Needs a GPU? |
|---|---|
| `01_eda.ipynb` | no |
| `02_build_training_data.ipynb` | no |
| `03_extend_tokenizer.ipynb` | no |
| `train_stages.py` | **yes** — 8–13 h for all three runs |
| `04_evaluate.ipynb` | **yes** — 45–90 min |
| `05_inference_and_export.ipynb` | yes |

Training itself is `train_stages.py`, not a notebook: it needs OOM recovery
and resumable staging that a kernel cannot give you.

Trained models are written to `artifacts/` and are **not** pushed to GitHub —
they are far too large. Copy them off the node yourself if you need to keep them.